In [1]:
import pandas as pd
import pickle, json
import numpy as np

df = pd.read_csv('bk_sentinel_verified.csv', low_memory=False)
model    = pickle.load(open('bk_best_model.pkl', 'rb'))
encoders = pickle.load(open('bk_label_encoders.pkl', 'rb'))
features = json.load(open('bk_feature_cols.json'))

# Get January 2026 High-risk accounts
df_jan = df[df['observation_month'] == '2026-01'].copy().reset_index(drop=True)
df_high = df_jan[df_jan['risk_state'] == 'High'].copy()

# Encode
df_high['segment_enc']   = encoders['segment'].transform(df_high['segment'].astype(str).fillna('UNKNOWN'))
df_high['loan_type_enc'] = encoders['loan_type'].transform(df_high['loan_type'].astype(str).fillna('UNKNOWN'))
for c in features:
    if c not in df_high.columns: df_high[c] = 0
df_high[features] = df_high[features].fillna(0)

X = df_high[features].values
proba = model.predict_proba(X)
preds = model.predict(X)

rl = {0:'Low', 1:'Medium', 2:'High', 3:'Default'}
df_high['pred'] = [rl[p] for p in preds]
df_high['p_default'] = proba[:, 3]

print('Prediction distribution for High-risk accounts:')
print(df_high['pred'].value_counts())
print()
print('P(Default) distribution:')
print(df_high['p_default'].describe().round(3))
print()
print('High accounts with P(Default) > 10%:')
print(df_high[df_high['p_default'] > 0.1][['loan_id','days_in_arrears','p_default']].sort_values('p_default', ascending=False).head(10))

Prediction distribution for High-risk accounts:
pred
High    124
Name: count, dtype: int64

P(Default) distribution:
count    124.000
mean       0.177
std        0.084
min        0.048
25%        0.110
50%        0.154
75%        0.240
max        0.421
Name: p_default, dtype: float64

High accounts with P(Default) > 10%:
       loan_id  days_in_arrears  p_default
3435  LN831063               83   0.420739
1291  LN369534               62   0.398969
3739  LN894243               54   0.388985
324   LN169132               80   0.377381
1910  LN501415               89   0.352689
2997  LN736648               77   0.333139
1911  LN501717               38   0.329810
3841  LN917511               52   0.312627
70    LN114028               88   0.300488
1205  LN353039               55   0.298517
